## Code de génération du graphe représentant le profile de vitesse saccadé ##

In [ ]:
"""
Profil de vitesse trapézoïdal 1/3-1/3-1/3 avec saccade de désalignement
Accouplement fourche-T -- modèle exact, périodique de 180°.

Modèle : un point (ergot) est entraîné à rayon fixe R autour de l'axe moteur.
L'axe tige, décalé radialement de e, "regarde" ce même point. Le rapport de
transmission instantané exact, sur un DEMI-tour, est :

    omega2/omega1 = R*(R + e*cos(alpha)) / (R^2 + e^2 + 2*R*e*cos(alpha))

La fourche en T (tige) étant symétrique (deux bras à 180°), ce rapport se
répète à l'identique tous les 180° de rotation moteur (et non tous les 360°) :
le second bras reproduit exactement le même relais que le premier. On observe
donc une discontinuité de vitesse (mais pas de position) à chaque demi-tour,
lorsque le bras actif change.

Aucune approximation n'est faite (pas de linéarisation, pas d'hypothèse de
petit angle) : e/R = 3.18/12.5 = 0.25 n'est pas négligeable.

Paramètres :
  - omega_max  = 60 tr/min
  - Durée T    = 6 s
  - e          = 3.18 mm (Delta h, désalignement réel maximal entre mouvements)
  - R          = 12.5 mm (rayon d'entraînement, demi-écart de la fourche)
"""

import numpy as np
import matplotlib.pyplot as plt

# ── Paramètres ────────────────────────────────────────────────────────────────
W_MAX = 60.0   # vitesse max (tr/min)
T_TOTAL = 6.0  # durée totale (s)
R = 12.5       # rayon d'entraînement (mm)
E = 3.18       # désalignement radial réel = Delta h (mm)
N = 3000       # points de simulation

# ── Fonctions ─────────────────────────────────────────────────────────────────
def trapezoidal(t, T, w_max):
    """Profil trapézoïdal 1/3-1/3-1/3."""
    t1, t2 = T / 3, 2 * T / 3
    return np.where(
        t <= t1, w_max * t / t1,
        np.where(t <= t2, w_max,
                 np.where(t <= T, w_max * (T - t) / (T - t2), 0.0))
    )

def with_jitter(w_ideal, e, R, t):
    """
    Vitesse avec saccade périodique due au désalignement, modèle exact,
    prolongé par périodicité de 180° (symétrie de la fourche en T) : le
    rapport est calculé avec l'angle réduit modulo 180°, ce qui produit un
    saut de vitesse (position continue) à chaque changement de bras actif.
    """
    omega_rad = w_ideal * 2 * np.pi / 60
    dt = t[1] - t[0]
    alpha = np.cumsum(omega_rad) * dt
    alpha_mod = np.mod(alpha, np.pi)

    ratio = R * (R + e * np.cos(alpha_mod)) / (R**2 + e**2 + 2 * R * e * np.cos(alpha_mod))
    return w_ideal * ratio

def eps_bounds(e, R):
    """Bornes exactes de l'erreur de vitesse relative (min en alpha=0, max en alpha=pi)."""
    return -e / (R + e) * 100, e / (R - e) * 100

eps_min, eps_max = eps_bounds(E, R)

# ── Calcul ────────────────────────────────────────────────────────────────────
t = np.linspace(0, T_TOTAL, N)
ideal = trapezoidal(t, T_TOTAL, W_MAX)
actual = with_jitter(ideal, E, R, t)

# ── Affichage console ─────────────────────────────────────────────────────────
print("=" * 55)
print("  Accouplement fourche-T -- analyse de désalignement (exact)")
print("=" * 55)
print(f"  Rayon d'entraînement    R       = {R:.1f} mm")
print(f"  Désalignement réel      e       = {E:.2f} mm (Delta h)")
print(f"  Erreur de vitesse       min/max = {eps_min:.1f}% / {eps_max:.1f}%")
print("=" * 55)

# ── Figure ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(10, 5))
fig.suptitle("Profil de vitesse trapézoïdal 1/3–1/3–1/3\n", fontsize=13, fontweight="bold")

ax1 = fig.add_subplot(111)
ax1.plot(t, ideal, color="#378ADD", lw=2.0, ls="--", label="Sans désalignement")
ax1.plot(t, actual, color="#D85A30", lw=1.4, label=f"Avec désalignement de {E:.2f} mm (cas réel)")

for x in (T_TOTAL / 3, 2 * T_TOTAL / 3, T_TOTAL):
    ax1.axvline(x, color="gray", lw=0.8, ls=":")

ax1.set_xlabel("Temps (s)")
ax1.set_ylabel("Vitesse (tr/min)")
ax1.legend(fontsize=9, loc="upper right")
ax1.set_xlim(0, T_TOTAL)
ax1.set_ylim(0, W_MAX * 1.5)
ax1.grid(True, lw=0.5, alpha=0.4)

fig.savefig("../../figures/Implementation/profil de vitesse trapezoidale avec saccade.png", dpi=100.7)
plt.show()
